# Solution C — APIM → Event Hub → Blob (Capture) 測試

**目的**：驗證 Solution C 完整 body 落地能力，並與 Solution ① 雙軌獨立。

**隔離原則**：
- 只有送出 header `X-Logging-Channel: solution-c` 的請求才會觸發 EH 寫入
- Solution ①（AppInsights）對所有流量持續啟用，互不干擾
- 每筆請求帶 `X-Run-Id`（=correlationId），用來事後在 EH/Blob/ADX 拼回完整對話

**前置**：
1. `.\scripts\deploy-eventhub-logging.ps1` ✅
2. `.\scripts\apply-eventhub-policy.ps1 -ApiName kunlenewfoundry01` ✅

In [ ]:
import os, time, json, subprocess
from openai import OpenAI
from getpass import getpass

# 認證重點：
# 1. product-scoped subscription key（非 master key），例如 kunlenewfoundry01-proj-default-ai-wl55p1p5w0
# 2. 此 API 用 header 名稱 `api-key`（不是 Ocp-Apim-Subscription-Key）
# 3. 用 OpenAI v1 client（Foundry endpoint 已經是 /openai/v1 路徑）

APIM_BASE_URL     = 'https://testaigw01.azure-api.net/kunlenewfoundry01/openai/v1/'
APIM_SUBSCRIPTION = os.environ.get('APIM_SUBSCRIPTION_KEY') or getpass('APIM Subscription Key (product-scoped, e.g. kunlenewfoundry01-proj-default-ai-wl55p1p5w0): ')
DEPLOYMENT_NAME   = 'Kimi-K2.5'

RUN_ID = f'solc-{int(time.time())}'
print('RUN_ID =', RUN_ID)

client = OpenAI(
    base_url        = APIM_BASE_URL,
    api_key         = 'unused',
    default_headers = {
        'api-key':           APIM_SUBSCRIPTION,
        'X-Logging-Channel': 'solution-c',
        'X-Run-Id':          RUN_ID,
    },
)
# ----- 驗證 helper：從 Blob Capture 拉對應 marker 的事件 -----
import subprocess, io, time as _t, sys, shutil
AZ = shutil.which('az') or shutil.which('az.cmd') or 'az'
RG = 'newfoundry01'
_STORAGE_CACHE = {}

def _get_storage_account():
    if 'name' in _STORAGE_CACHE: return _STORAGE_CACHE['name']
    out = subprocess.run([AZ,'storage','account','list','-g',RG,'-o','json'], capture_output=True, text=True)
    name = [s['name'] for s in json.loads(out.stdout) if s['name'].startswith('staigwc')][0]
    _STORAGE_CACHE['name'] = name
    return name

def _list_blobs(since_iso=None):
    sa = _get_storage_account()
    # Server-side prefix narrows blob list (path encodes date) — must NOT include partition since EH spreads across partitions
    prefix = None
    if since_iso:
        # use up to the day part of since_iso to limit scope (path encodes Y/M/D)
        try:
            d = since_iso[:10].replace('-', '/')  # 2026-04-19 -> 2026/04/19
            prefix = None  # date is per-partition so we can't easily prefix; rely on --num-results *
        except Exception:
            prefix = None
    cmd = [AZ,'storage','blob','list','--account-name',sa,'--container-name','capture','--auth-mode','login','--num-results','*','-o','json']
    out = subprocess.run(cmd, capture_output=True, text=True)
    if out.returncode != 0: return []
    blobs = json.loads(out.stdout)
    blobs = [b for b in blobs if int(b['properties']['contentLength']) > 1000]
    if since_iso:
        blobs = [b for b in blobs if b['properties']['lastModified'] >= since_iso]
    return sorted(blobs, key=lambda b: b['properties']['lastModified'], reverse=True)

def _download_blob(name):
    import tempfile, os as _os
    sa = _get_storage_account()
    fd, path = tempfile.mkstemp(suffix='.avro'); _os.close(fd); _os.remove(path)
    subprocess.run([AZ,'storage','blob','download','--account-name',sa,'--container-name','capture','--name',name,'--auth-mode','login','--no-progress','-f',path], capture_output=True)
    with open(path,'rb') as f: raw = f.read()
    try: _os.remove(path)
    except: pass
    return raw

def _parse_avro(raw):
    import fastavro
    events = []
    for rec in fastavro.reader(io.BytesIO(raw)):
        b = rec.get('Body')
        if isinstance(b,(bytes,bytearray)): b = b.decode('utf-8','replace')
        try: events.append(json.loads(b))
        except: events.append({'_raw': b})
    return events

def verify_marker(marker, expect_found=True, max_wait_sec=180, poll_sec=15, since_iso=None):
    """輪詢 blob capture，找到包含 marker 的事件後印出（input body + output body + token usage）。
    expect_found=False: 用於 TC-C4 對照組，預期找不到。"""
    print(f'[verify] marker = {marker}')
    print(f'[verify] storage = {_get_storage_account()} / capture')
    print(f'[verify] expect_found = {expect_found}, max_wait = {max_wait_sec}s')
    deadline = _t.time() + max_wait_sec
    matching = []
    matched_blobs = set()
    expected_kinds = {'summary','request-body','response-body'}
    while _t.time() < deadline:
        blobs = _list_blobs(since_iso=since_iso)
        # Aggregate hits from ALL blobs in the window (EH partitions events across blobs)
        agg = []; agg_blobs = set()
        for b in blobs:
            raw = _download_blob(b['name'])
            evts = _parse_avro(raw)
            hits = [e for e in evts if marker in json.dumps(e, ensure_ascii=False)]
            if hits:
                agg.extend(hits); agg_blobs.add(b['name'])
        kinds = {e.get('kind') for e in agg if isinstance(e,dict)}
        # Stop early if we have all 3 event kinds
        if expected_kinds.issubset(kinds):
            matching = agg; matched_blobs = agg_blobs; break
        # Or stop if we have something and have given Capture another full window
        if agg and _t.time() - (deadline - max_wait_sec) > 75:
            matching = agg; matched_blobs = agg_blobs; break
        if not expect_found and _t.time() - (deadline - max_wait_sec) > 60:
            break
        remain = int(deadline - _t.time())
        print(f'  ...have {len(agg)} event(s) so far {sorted(kinds)}, sleeping {poll_sec}s (remain {remain}s)')
        _t.sleep(poll_sec)
    if expect_found:
        if not matching:
            print(f'❌ FAIL: marker {marker} 在 {max_wait_sec}s 內找不到於 capture container')
            return None
        print(f'✅ found {len(matching)} event(s) across {len(matched_blobs)} blob(s):')
        for _b in sorted(matched_blobs): print(f'    - {_b}')
        print()
        # ---- separate by kind ----
        summaries  = [e for e in matching if e.get('kind')=='summary']
        req_evts   = [e for e in matching if e.get('kind')=='request-body']
        resp_evts  = [e for e in matching if e.get('kind')=='response-body']
        # Sort summaries by timestamp so retry chronology is clear
        summaries.sort(key=lambda x: x.get('timestamp',''))
        # ===== ATTEMPT TIMELINE (handles APIM <retry> on 429/5xx) =====
        if len(summaries) > 1:
            print(f'=== ATTEMPT TIMELINE — {len(summaries)} summaries (APIM <retry> recovered from non-2xx) ===')
            for i,s in enumerate(summaries):
                print(f'  attempt #{i+1}: status={s.get("status"):>3}  dur={s.get("durationMs")}ms  reqLen={s.get("requestLength")}  respLen={s.get("responseLength")}  reqChunks={s.get("requestChunks")}  respChunks={s.get("responseChunks")}  ts={s.get("timestamp")}')
            print()
        # ===== Pick the attempt to deep-parse: prefer 2xx, else last =====
        if summaries:
            ok = [s for s in summaries if 200 <= (s.get('status') or 0) < 300]
            chosen_summary = ok[-1] if ok else summaries[-1]
            print('=== SUMMARY (chosen attempt for body parsing) ===')
            for k in ('method','url','status','durationMs','requestLength','responseLength','requestChunks','responseChunks','subscriptionId','clientIp','timestamp'):
                print(f'  {k:>15}: {chosen_summary.get(k)}')
            print()
        else:
            chosen_summary = None
        # ===== Reassemble request-body for chosen attempt =====
        # Bodies don't carry attempt index. Strategy: dedupe payloads (request body usually identical across retries).
        # If multiple distinct payloads exist, prefer the one whose total length matches summary.requestLength.
        def _reassemble(events, expected_total_len=None, label=''):
            if not events: return None, None, None
            # group by chunkIndex -> set of distinct payloads at that index
            by_idx = {}
            for e in events:
                by_idx.setdefault(e.get('chunkIndex'), []).append(e.get('payload',''))
            # build candidate full bodies: cartesian on idx 0..N-1 is overkill; assume payload at each idx is same OR pick one matching expected_total_len
            chunkTotal = events[0].get('chunkTotal')
            indices = sorted(by_idx.keys())
            integrity_ok = (chunkTotal == len(indices) and indices == list(range(chunkTotal)))
            # For each idx, dedupe payloads
            distinct_per_idx = {i: list({p for p in by_idx[i]}) for i in indices}
            # If any idx has >1 distinct payload, try to pick the combination matching expected length
            chosen_payloads = []
            for i in indices:
                opts = distinct_per_idx[i]
                chosen_payloads.append(opts[0] if len(opts) == 1 else None)
            if any(p is None for p in chosen_payloads) and expected_total_len:
                # pick per-idx the payload that, when summed with others' best, is closest to expected
                for j,i in enumerate(indices):
                    if chosen_payloads[j] is not None: continue
                    # pick payload whose length, when added to already-fixed lengths, fits remaining budget
                    fixed_len = sum(len(p) for p in chosen_payloads if p is not None)
                    remaining = expected_total_len - fixed_len
                    # for unfilled indices count - 1 others, naively pick payload nearest to (remaining / unfilled)
                    unfilled = sum(1 for p in chosen_payloads if p is None)
                    target = remaining / max(unfilled,1)
                    chosen_payloads[j] = min(distinct_per_idx[i], key=lambda p: abs(len(p) - target))
            full = ''.join(chosen_payloads)
            integrity = '✅ COMPLETE' if integrity_ok else f'❌ MISSING (got idx {indices}, expected 0..{chunkTotal-1})'
            distinct_summary = ' · '.join(f'idx{i}:{len(distinct_per_idx[i])} variant' + ('s' if len(distinct_per_idx[i])>1 else '') for i in indices)
            return full, f'{len(events)} events ({distinct_summary}) · {chunkTotal} chunks · {integrity}', chunkTotal
        # ===== REQUEST BODY =====
        if req_evts:
            exp_req_len = chosen_summary.get('requestLength') if chosen_summary else None
            req_full, req_meta, req_total = _reassemble(req_evts, exp_req_len, 'request')
            len_check = ''
            if exp_req_len is not None:
                len_check = ' · ' + ('✅ length matches summary' if exp_req_len == len(req_full) else f'❌ length {len(req_full)} ≠ summary {exp_req_len}')
            print(f'=== REQUEST BODY  ({req_meta}{len_check}) ===')
            print(f'  reassembled total: {len(req_full)} chars')
            try:
                pj = json.loads(req_full)
                print(f'  model={pj.get("model")}  stream={pj.get("stream")}  max_tokens={pj.get("max_tokens")}')
                for m in pj.get('messages',[]):
                    c = (m.get('content') or '')
                    print(f'  [{m.get("role")}] ({len(c)} chars) {c[:300]}{"..." if len(c)>300 else ""}')
            except Exception as ex:
                print(f'  (raw / parse failed: {ex}): {req_full[:400]}')
            print()
        # ===== RESPONSE BODY =====
        if resp_evts:
            exp_resp_len = chosen_summary.get('responseLength') if chosen_summary else None
            resp_full, resp_meta, resp_total = _reassemble(resp_evts, exp_resp_len, 'response')
            len_check = ''
            if exp_resp_len is not None:
                len_check = ' · ' + ('✅ length matches summary' if exp_resp_len == len(resp_full) else f'❌ length {len(resp_full)} ≠ summary {exp_resp_len}')
            print(f'=== RESPONSE BODY ({resp_meta}{len_check}) ===')
            print(f'  reassembled total: {len(resp_full)} chars')
            if resp_full.lstrip().startswith('data:'):
                content_acc = []; reasoning_acc = []; usage = None; sse_count = 0; finish = None
                for line in resp_full.splitlines():
                    if not line.startswith('data:'): continue
                    data = line[5:].strip()
                    if data == '[DONE]' or not data: continue
                    sse_count += 1
                    try:
                        d = json.loads(data)
                        for ch in d.get('choices',[]) or []:
                            delta = ch.get('delta',{}) or {}
                            if delta.get('content'): content_acc.append(delta['content'])
                            if delta.get('reasoning_content'): reasoning_acc.append(delta['reasoning_content'])
                            if ch.get('finish_reason'): finish = ch['finish_reason']
                        if d.get('usage'): usage = d['usage']
                    except: pass
                content_full = ''.join(content_acc)
                reasoning_full = ''.join(reasoning_acc)
                print(f'  [streaming] {sse_count} SSE events · finish_reason={finish}')
                print(f'  reasoning_content: {len(reasoning_full)} chars')
                if reasoning_full: print(f'    preview: {reasoning_full[:300]}{"..." if len(reasoning_full)>300 else ""}')
                print(f'  content:           {len(content_full)} chars')
                if content_full:   print(f'    preview: {content_full[:300]}{"..." if len(content_full)>300 else ""}')
                print(f'  usage (from final SSE chunk): {usage}')
            else:
                try:
                    pj = json.loads(resp_full)
                    ch0 = (pj.get('choices') or [{}])[0]
                    msg = ch0.get('message',{}) or {}
                    content = msg.get('content') or ''
                    reasoning = msg.get('reasoning_content') or ''
                    print(f'  model={pj.get("model")}  finish_reason={ch0.get("finish_reason")}')
                    print(f'  reasoning_content: {len(reasoning)} chars')
                    if reasoning: print(f'    preview: {reasoning[:300]}{"..." if len(reasoning)>300 else ""}')
                    print(f'  content:           {len(content)} chars')
                    if content:   print(f'    preview: {content[:300]}{"..." if len(content)>300 else ""}')
                    print(f'  usage: {pj.get("usage")}   ← ✅ 完整 prompt/completion/total tokens')
                except Exception as ex:
                    print(f'  (parse failed: {ex}) raw[0:300]: {resp_full[:300]}')
            print()
        return matching
    else:
        if matching:
            print(f'❌ FAIL: 對照組 marker {marker} **不應**出現在 EH，卻找到 {len(matching)} 筆於 {len(matched_blobs)} blob')
            return matching
        print(f'✅ PASS: 對照組 marker {marker} 確實**未**出現在 EH（隔離有效）')
        return []

# 記錄測試起始時間，避免 verify 時掃到很舊的 blob
RUN_START_ISO = _t.strftime('%Y-%m-%dT%H:%M:%SZ', _t.gmtime(_t.time() - 60))
print('RUN_START_ISO:', RUN_START_ISO)


## TC-C1 — Non-streaming 短回覆（驗 happy path）

In [ ]:
import json
marker = f'{RUN_ID}-c1'
_user_prompt = f'[{marker}] 用一句話說明 Event Hub'
_req_body = {
    'model': DEPLOYMENT_NAME,
    'messages': [{'role':'user','content': _user_prompt}],
    'max_tokens': 2000,
}
resp = client.chat.completions.create(
    **_req_body,
    extra_headers = {'X-Run-Id': marker},
)
print('Marker          :', marker)
print('Status          :', 'OK')
print()
print('--- Request body (will be captured by Solution C as request-body) ---')
print(json.dumps(_req_body, ensure_ascii=False, indent=2))
print()
print('--- Response usage (compare with usage block in blob) ---')
print('  prompt_tokens     :', resp.usage.prompt_tokens)
print('  completion_tokens :', resp.usage.completion_tokens)
print('  total_tokens      :', resp.usage.total_tokens)
print()
print('--- Response body (will be captured by Solution C as response-body) ---')
_msg = resp.choices[0].message
_content = _msg.content or ''
_reason  = getattr(_msg, 'reasoning_content', None) or ''
print('finish_reason     :', resp.choices[0].finish_reason)
print('content length    :', len(_content), 'chars')
print('reasoning length  :', len(_reason), 'chars')
print()
print('--- content (full) ---')
print(_content if _content else '(empty — reasoning model put text in reasoning_content)')
print()
print('--- reasoning_content (full) ---')
print(_reason if _reason else '(none)')
print()
print('>>> 上面這份 request + response 的 token 數與內文，稍後可在下一個 cell 拉到的 blob avro 內容中找到對應的 request-body / response-body / summary。')


### 🔍 驗證 TC-C1：等 Capture flush 後從 Blob 拉回 — 確認 input message + output content + **完整 token usage**

In [ ]:
verify_marker(marker, expect_found=True, max_wait_sec=180, since_iso=RUN_START_ISO)

## TC-C2 — 大型回覆（驗證超過 256KB 不被截斷，多 chunk）

In [ ]:
import json
marker = f'{RUN_ID}-c2'
_user_prompt = f'[{marker}] 用 reasoning 詳細解釋 Kubernetes 從零到生產所有概念，至少 8000 字繁中'
_req_body = {
    'model': DEPLOYMENT_NAME,
    'messages': [{'role':'user','content': _user_prompt}],
    'max_tokens': 8000,
}
resp = client.chat.completions.create(
    **_req_body,
    extra_headers = {'X-Run-Id': marker},
)
print('Marker          :', marker)
print('Status          :', 'OK')
print()
print('--- Request body (will be captured by Solution C as request-body) ---')
print(json.dumps(_req_body, ensure_ascii=False, indent=2))
print()
_msg = resp.choices[0].message
_content = _msg.content or ''
_reason  = getattr(_msg, 'reasoning_content', None) or ''
_text    = _content or _reason
_bytes   = len(_text.encode('utf-8'))
print('--- Response usage (compare with usage block in blob) ---')
print('  prompt_tokens     :', resp.usage.prompt_tokens)
print('  completion_tokens :', resp.usage.completion_tokens)
print('  total_tokens      :', resp.usage.total_tokens)
print('  finish_reason     :', resp.choices[0].finish_reason)
print()
print('--- Response body sizing ---')
print('  content length    :', len(_content), 'chars')
print('  reasoning length  :', len(_reason), 'chars')
print('  primary text bytes:', _bytes, 'bytes (UTF-8)  ← 若 >256KB 則方案①會被截，方案C完整保留')
print()
print('--- content (full) ---')
print(_content if _content else '(empty — reasoning model put text in reasoning_content)')
print()
print('--- reasoning_content (full) ---')
print(_reason if _reason else '(none)')
print()
print('>>> 上面這份 request + 完整 response 內文 + token 數，稍後在 verify cell 拉到的 blob avro 內容中應全部一致對應到 request-body / response-body / summary。')


### 🔍 驗證 TC-C2：大型回覆 — 確認 multi-chunk reassembly + **token usage 完整**

In [ ]:
verify_marker(marker, expect_found=True, max_wait_sec=180, since_iso=RUN_START_ISO)

## TC-C3 — Streaming SSE（驗證 R4 場景下方案 C 是否拿得到完整 body）

In [ ]:
import json
marker = f'{RUN_ID}-c3'
_user_prompt = f'[{marker}] streaming 解釋 reasoning model 如何推理'
_req_body = {
    'model': DEPLOYMENT_NAME,
    'messages': [{'role':'user','content': _user_prompt}],
    'max_tokens': 4000,
    'stream': True,
    'stream_options': {'include_usage': True},
}
print('--- Request body (will be captured by Solution C as request-body) ---')
print(json.dumps({k:v for k,v in _req_body.items()}, ensure_ascii=False, indent=2))
print()
stream = client.chat.completions.create(
    **_req_body,
    extra_headers = {'X-Run-Id': marker},
)
_content_chunks = []
_reason_chunks  = []
_usage = None
_finish = None
for chunk in stream:
    if chunk.choices:
        delta = chunk.choices[0].delta
        if getattr(delta,'content',None):
            _content_chunks.append(delta.content)
        if getattr(delta,'reasoning_content',None):
            _reason_chunks.append(delta.reasoning_content)
        if chunk.choices[0].finish_reason:
            _finish = chunk.choices[0].finish_reason
    if getattr(chunk,'usage',None):
        _usage = chunk.usage
_content = ''.join(_content_chunks)
_reason  = ''.join(_reason_chunks)
print('Marker          :', marker)
print('Status          :', 'OK (streaming)')
print()
print('--- Response usage (compare with usage in final SSE chunk in blob) ---')
if _usage:
    print('  prompt_tokens     :', _usage.prompt_tokens)
    print('  completion_tokens :', _usage.completion_tokens)
    print('  total_tokens      :', _usage.total_tokens)
else:
    print('  (no usage in stream — include_usage may be unsupported)')
print('  finish_reason     :', _finish)
print()
print('--- Reassembled streaming response ---')
print('  content length    :', len(_content), 'chars')
print('  reasoning length  :', len(_reason), 'chars')
print()
print('--- content (full, reassembled from SSE deltas) ---')
print(_content if _content else '(empty — reasoning model put text in reasoning_content)')
print()
print('--- reasoning_content (full, reassembled from SSE deltas) ---')
print(_reason if _reason else '(none)')
print()
print('>>> verify cell 拿到的 response-body payload 是 SSE 原文，會被 helper 自動拼回；usage 應在最後一個 data: 區塊。')


### 🔍 驗證 TC-C3：Streaming — 確認 SSE 完整捕捉 + 從最後一筆 chunk 取出 usage

In [ ]:
verify_marker(marker, expect_found=True, max_wait_sec=180, since_iso=RUN_START_ISO)

## TC-C4 — 對照組：**不帶** header（應只進 Solution ①，**不**進 EH）

In [ ]:
# 對照組：故意不帶 X-Logging-Channel header → Solution C policy 應 skip 寫 EH
import json
marker = f'{RUN_ID}-c4-controlgroup'
_user_prompt = f'[{marker}] hello'
_req_body = {
    'model': DEPLOYMENT_NAME,
    'messages': [{'role':'user','content': _user_prompt}],
    'max_tokens': 50,
}
control = OpenAI(
    base_url        = APIM_BASE_URL,
    api_key         = 'unused',
    default_headers = {'api-key': APIM_SUBSCRIPTION},  # 故意不帶 X-Logging-Channel
)
resp = control.chat.completions.create(**_req_body)
_msg = resp.choices[0].message
_content = _msg.content or ''
_reason  = getattr(_msg, 'reasoning_content', None) or ''
print('Marker          :', marker)
print('Status          :', 'OK')
print()
print('--- Request body (NOT expected in Solution C blob — control group) ---')
print(json.dumps(_req_body, ensure_ascii=False, indent=2))
print()
print('--- Response usage ---')
print('  prompt_tokens     :', resp.usage.prompt_tokens)
print('  completion_tokens :', resp.usage.completion_tokens)
print('  total_tokens      :', resp.usage.total_tokens)
print('  finish_reason     :', resp.choices[0].finish_reason)
print()
print('--- content (full) ---')
print(_content if _content else '(empty)')
print()
print('--- reasoning_content (full) ---')
print(_reason if _reason else '(none)')
print()
print('>>> 預期：verify cell 在 EH/Blob 中**找不到**這個 marker；它只會出現在 Solution ① AppInsights traces。')


### 🔍 驗證 TC-C4 對照組：marker **不應**出現在 EH（隔離驗證）

In [ ]:
verify_marker(marker, expect_found=False, max_wait_sec=180, since_iso=RUN_START_ISO)

## ✅ 驗收標準

| 項目 | 預期 |
|---|---|
| TC-C1 (短) | EH 內找得到 1 summary + 1 request-body chunk + 1 response-body chunk |
| TC-C2 (大) | response-body 拆 ≥ 2 chunk，總 length 大於 256KB（Solution ① 在此會被截）|
| TC-C3 (stream) | 仍能拿到完整 body（驗證 R4 在 Capture 路徑下表現）|
| TC-C4 (對照) | EH/Blob 內**沒有** `c4-controlgroup` 字樣 → 證明 header 隔離有效 |
| Solution ① | 全部 4 個 marker 都應在 AppInsights AppDependencies 找得到 → 證明雙軌獨立 |

## 後續查詢進階

如要長期 ad-hoc 查詢，建議掛 ADX external table 指向 capture container（範例：`kql/queries-eventhub.kql` C1-C5）。